In [39]:
!pip install -q langchain langgraph chromadb sentence-transformers pypdf

In [40]:
# ============================================
# STEP 1.2 - Import Libraries
# ============================================

import os
import re
import json
from pathlib import Path

from google.colab import files

from pypdf import PdfReader

print("Libraries imported successfully! ✅")

Libraries imported successfully! ✅


In [41]:
# ============================================
# STEP 1.3 - Project Configuration
# ============================================

PROJECT_NAME = "Agentic AI Research & Document Assistant"

DATA_DIR = Path("/content/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_NAME)
print("Data directory:", DATA_DIR)

Project: Agentic AI Research & Document Assistant
Data directory: /content/data


In [42]:
# ============================================
# STEP 1.4 - Upload Research PDF
# ============================================

print("Please upload your research PDF...")

uploaded = files.upload()

print("\nUploaded files:")

for filename in uploaded.keys():
    print("✅", filename)

Please upload your research PDF...


Saving 1.pdf to 1.pdf

Uploaded files:
✅ 1.pdf


In [43]:
# ============================================
# STEP 1.5 - Save Uploaded PDF
# ============================================

uploaded_filename = list(uploaded.keys())[0]

source_path = Path("/content") / uploaded_filename
destination_path = DATA_DIR / uploaded_filename

source_path.rename(destination_path)

print("PDF saved successfully! ✅")
print("File:", destination_path)

PDF saved successfully! ✅
File: /content/data/1.pdf


In [44]:
# ============================================
# STEP 1.6 - Validate PDF
# ============================================

pdf_path = destination_path

reader = PdfReader(str(pdf_path))

print("PDF loaded successfully! ✅")
print("File name:", pdf_path.name)
print("Number of pages:", len(reader.pages))

PDF loaded successfully! ✅
File name: 1.pdf
Number of pages: 17


In [45]:
# ============================================
# STEP 1.7 - Extract Text From First Page
# ============================================

first_page_text = reader.pages[0].extract_text()

print("First page text:\n")
print(first_page_text[:3000])

First page text:

Modern
Construction
Lean Project Delivery
and Integrated Practices
Lincoln H. Forbes
Syed M. Ahmed
CRC Press
Taylor & Francis Croup
Boca Raton London New York
CRC Press is an imprint of the
Taylor & Francis Croup, an informs business



In [46]:
# ============================================
# STEP 2.1 - Extract Text From All Pages
# ============================================

documents = []

for page_number, page in enumerate(reader.pages, start=1):

    text = page.extract_text()

    if text:
        documents.append({
            "text": text,
            "page": page_number,
            "source": pdf_path.name
        })

print("Total pages extracted:", len(documents))
print()

# Show first document
print("First document:")
print("Source:", documents[0]["source"])
print("Page:", documents[0]["page"])
print("Text preview:")
print(documents[0]["text"][:1000])

Total pages extracted: 17

First document:
Source: 1.pdf
Page: 1
Text preview:
Modern
Construction
Lean Project Delivery
and Integrated Practices
Lincoln H. Forbes
Syed M. Ahmed
CRC Press
Taylor & Francis Croup
Boca Raton London New York
CRC Press is an imprint of the
Taylor & Francis Croup, an informs business



In [47]:
# ============================================
# STEP 2.2 - Check Extracted Documents
# ============================================

for doc in documents[:5]:
    print(
        f"Page {doc['page']} | "
        f"Characters: {len(doc['text'])} | "
        f"Source: {doc['source']}"
    )

Page 1 | Characters: 234 | Source: 1.pdf
Page 2 | Characters: 1460 | Source: 1.pdf
Page 3 | Characters: 1426 | Source: 1.pdf
Page 4 | Characters: 1205 | Source: 1.pdf
Page 5 | Characters: 1512 | Source: 1.pdf


In [48]:
# ============================================
# STEP 2.3 - Clean Extracted Text
# ============================================

def clean_text(text):
    # Replace multiple spaces with a single space
    text = re.sub(r'\s+', ' ', text)

    # Remove leading/trailing spaces
    text = text.strip()

    return text


for doc in documents:
    doc["text"] = clean_text(doc["text"])


print("Text cleaning completed! ✅")
print()
print(documents[0]["text"][:1500])

Text cleaning completed! ✅

Modern Construction Lean Project Delivery and Integrated Practices Lincoln H. Forbes Syed M. Ahmed CRC Press Taylor & Francis Croup Boca Raton London New York CRC Press is an imprint of the Taylor & Francis Croup, an informs business


In [49]:
# ============================================
# STEP 2.4 - Check Document Structure
# ============================================

print("Document structure:")
print(documents[0].keys())

print("\nExample:")
print({
    "source": documents[0]["source"],
    "page": documents[0]["page"],
    "text_preview": documents[0]["text"][:200]
})

Document structure:
dict_keys(['text', 'page', 'source'])

Example:
{'source': '1.pdf', 'page': 1, 'text_preview': 'Modern Construction Lean Project Delivery and Integrated Practices Lincoln H. Forbes Syed M. Ahmed CRC Press Taylor & Francis Croup Boca Raton London New York CRC Press is an imprint of the Taylor & F'}


In [50]:
# ============================================
# STEP 3.1 - Text Chunking Function
# ============================================

def create_chunks(documents, chunk_size=500, overlap=50):
    """
    Convert page-wise documents into smaller text chunks.

    Parameters:
        documents: list of dictionaries containing text, page, source
        chunk_size: approximate number of words per chunk
        overlap: number of overlapping words between chunks

    Returns:
        List of chunk dictionaries
    """

    chunks = []

    for doc in documents:

        text = doc["text"]
        words = text.split()

        # Skip empty pages
        if not words:
            continue

        start = 0

        while start < len(words):

            end = start + chunk_size

            chunk_words = words[start:end]
            chunk_text = " ".join(chunk_words)

            chunks.append({
                "text": chunk_text,
                "page": doc["page"],
                "source": doc["source"]
            })

            # Move forward while keeping overlap
            start += chunk_size - overlap

    return chunks


print("Chunking function created successfully! ✅")

Chunking function created successfully! ✅


In [51]:
# ============================================
# STEP 3.2 - Create Document Chunks
# ============================================

chunks = create_chunks(
    documents,
    chunk_size=500,
    overlap=50
)

print("Chunking completed successfully! ✅")
print("Total chunks:", len(chunks))

Chunking completed successfully! ✅
Total chunks: 17


In [52]:
# ============================================
# STEP 3.3 - Inspect First Chunk
# ============================================

first_chunk = chunks[0]

print("First Chunk")
print("=" * 60)

print("Source:", first_chunk["source"])
print("Page:", first_chunk["page"])
print("Text:")
print(first_chunk["text"])

First Chunk
Source: 1.pdf
Page: 1
Text:
Modern Construction Lean Project Delivery and Integrated Practices Lincoln H. Forbes Syed M. Ahmed CRC Press Taylor & Francis Croup Boca Raton London New York CRC Press is an imprint of the Taylor & Francis Croup, an informs business


In [53]:
# ============================================
# STEP 3.4 - Inspect First 5 Chunks
# ============================================

for i, chunk in enumerate(chunks[:5]):

    print(f"\n{'=' * 70}")
    print(f"CHUNK {i + 1}")
    print(f"{'=' * 70}")

    print("Source:", chunk["source"])
    print("Page:", chunk["page"])
    print("Characters:", len(chunk["text"]))
    print("Preview:", chunk["text"][:300])


CHUNK 1
Source: 1.pdf
Page: 1
Characters: 233
Preview: Modern Construction Lean Project Delivery and Integrated Practices Lincoln H. Forbes Syed M. Ahmed CRC Press Taylor & Francis Croup Boca Raton London New York CRC Press is an imprint of the Taylor & Francis Croup, an informs business

CHUNK 2
Source: 1.pdf
Page: 2
Characters: 1459
Preview: Contents Comments by Greg Howell, Co-Founder of the Lean Construction Institute xxv Preface xxvii What's in This Book xxviii Acknowledgments xxxi Authors xxxiii 1 Overview of the Construction Industry 1 Background on Industry Performance 1 Reasons for Low Productivity 2 Need for New Approaches to Co

CHUNK 3
Source: 1.pdf
Page: 3
Characters: 1425
Preview: x Contents Growing Emergence of Subcontracting 29 Slow Adoption of Innovation 29 Lack of Benchmarking 29 Crisis Orientation 29 Labor Shortages 30 Project Uniqueness 30 Technology Impacts 30 Real Wage Trends 31 Inadequate Construction Training 31 Productivity Ratios 32 Total Productivity 32 Cons

In [54]:
# ============================================
# STEP 3.5 - Validate Chunk Metadata
# ============================================

print("Chunk keys:")
print(chunks[0].keys())

print("\nFirst chunk metadata:")
print({
    "source": chunks[0]["source"],
    "page": chunks[0]["page"]
})

Chunk keys:
dict_keys(['text', 'page', 'source'])

First chunk metadata:
{'source': '1.pdf', 'page': 1}


In [55]:
# ============================================
# STEP 3.6 - Chunk Statistics
# ============================================

chunk_word_counts = [
    len(chunk["text"].split())
    for chunk in chunks
]

print("Total chunks:", len(chunks))
print("Minimum words:", min(chunk_word_counts))
print("Maximum words:", max(chunk_word_counts))
print("Average words:", round(
    sum(chunk_word_counts) / len(chunk_word_counts), 2
))

Total chunks: 17
Minimum words: 39
Maximum words: 240
Average words: 195.12


In [56]:
# ============================================
# STEP 3.7 - Add Unique Chunk IDs
# ============================================

for i, chunk in enumerate(chunks):

    chunk["id"] = f"chunk_{i}"

print("Chunk IDs added successfully! ✅")

print(chunks[0])

Chunk IDs added successfully! ✅
{'text': 'Modern Construction Lean Project Delivery and Integrated Practices Lincoln H. Forbes Syed M. Ahmed CRC Press Taylor & Francis Croup Boca Raton London New York CRC Press is an imprint of the Taylor & Francis Croup, an informs business', 'page': 1, 'source': '1.pdf', 'id': 'chunk_0'}


In [57]:
# ============================================
# STEP 3.8 - Final Chunk Validation
# ============================================

print("============================================")
print("       CHUNKING VALIDATION")
print("============================================")

print("Total documents/pages:", len(documents))
print("Total chunks:", len(chunks))

print("\nFirst chunk:")
print("ID:", chunks[0]["id"])
print("Source:", chunks[0]["source"])
print("Page:", chunks[0]["page"])
print("Words:", len(chunks[0]["text"].split()))

print("\nLast chunk:")
print("ID:", chunks[-1]["id"])
print("Source:", chunks[-1]["source"])
print("Page:", chunks[-1]["page"])
print("Words:", len(chunks[-1]["text"].split()))

print("\n✅ STEP 3 COMPLETED SUCCESSFULLY!")

       CHUNKING VALIDATION
Total documents/pages: 17
Total chunks: 17

First chunk:
ID: chunk_0
Source: 1.pdf
Page: 1
Words: 39

Last chunk:
ID: chunk_16
Source: 1.pdf
Page: 17
Words: 58

✅ STEP 3 COMPLETED SUCCESSFULLY!


In [58]:
# ============================================
# STEP 3.9 - PDF Extraction Diagnostic
# ============================================

for doc in documents[:3]:

    print("=" * 80)
    print("PAGE:", doc["page"])
    print("CHARACTERS:", len(doc["text"]))
    print("WORDS:", len(doc["text"].split()))
    print("=" * 80)

    print(doc["text"][:2000])
    print()

PAGE: 1
CHARACTERS: 233
WORDS: 39
Modern Construction Lean Project Delivery and Integrated Practices Lincoln H. Forbes Syed M. Ahmed CRC Press Taylor & Francis Croup Boca Raton London New York CRC Press is an imprint of the Taylor & Francis Croup, an informs business

PAGE: 2
CHARACTERS: 1459
WORDS: 200
Contents Comments by Greg Howell, Co-Founder of the Lean Construction Institute xxv Preface xxvii What's in This Book xxviii Acknowledgments xxxi Authors xxxiii 1 Overview of the Construction Industry 1 Background on Industry Performance 1 Reasons for Low Productivity 2 Need for New Approaches to Construction 3 Causes of Poor Construction Industry Performance 5 Categories of Construction 7 Who Are the Parties Involved in Construction? 8 Project Delivery Methods 8 Design-Bid-Build Contracts 9 Design-Build Projects 11 Engineer-Procure-Construct (EPC) Projects 13 Design-Construction Management (CM) Contracts 13 Design-Agency CM Contracts 14 CM-at-Risk Contracts 15 Fast-Track Construction 1

In [59]:
# ============================================
# STEP 3.10 - Improved Document Chunking
# ============================================

def create_improved_chunks(documents, chunk_size=500, overlap=50):

    chunks = []

    # Combine all pages while keeping page information
    all_words = []

    for doc in documents:

        words = doc["text"].split()

        for word in words:
            all_words.append({
                "word": word,
                "page": doc["page"],
                "source": doc["source"]
            })

    start = 0
    chunk_id = 0

    while start < len(all_words):

        end = min(start + chunk_size, len(all_words))

        chunk_items = all_words[start:end]

        chunk_text = " ".join(
            item["word"] for item in chunk_items
        )

        start_page = chunk_items[0]["page"]
        end_page = chunk_items[-1]["page"]
        source = chunk_items[0]["source"]

        chunks.append({
            "id": f"chunk_{chunk_id}",
            "text": chunk_text,
            "source": source,
            "start_page": start_page,
            "end_page": end_page,
            "word_count": len(chunk_items)
        })

        chunk_id += 1

        # Move forward while keeping overlap
        start += chunk_size - overlap

    return chunks


# Create improved chunks
chunks = create_improved_chunks(
    documents,
    chunk_size=500,
    overlap=50
)

print("Improved chunking completed! ✅")
print("Total chunks:", len(chunks))

Improved chunking completed! ✅
Total chunks: 8


In [60]:
# ============================================
# STEP 3.11 - Inspect Improved Chunks
# ============================================

for chunk in chunks[:5]:

    print("=" * 80)
    print("ID:", chunk["id"])
    print("Source:", chunk["source"])
    print("Pages:", chunk["start_page"], "→", chunk["end_page"])
    print("Words:", chunk["word_count"])
    print()
    print(chunk["text"][:500])
    print()

ID: chunk_0
Source: 1.pdf
Pages: 1 → 4
Words: 500

Modern Construction Lean Project Delivery and Integrated Practices Lincoln H. Forbes Syed M. Ahmed CRC Press Taylor & Francis Croup Boca Raton London New York CRC Press is an imprint of the Taylor & Francis Croup, an informs business Contents Comments by Greg Howell, Co-Founder of the Lean Construction Institute xxv Preface xxvii What's in This Book xxviii Acknowledgments xxxi Authors xxxiii 1 Overview of the Construction Industry 1 Background on Industry Performance 1 Reasons for Low Productivi

ID: chunk_1
Source: 1.pdf
Pages: 4 → 6
Words: 500

Traditional Construction 59 Barriers to Applying Manufacturing Methods to Construction 60 Characteristics of Lean Construction 60 Lean Principles 61 Value 61 Value Stream 62 Value Stream Mapping 62 Flow 62 Pull 62 Perfection 63 Systems Perspective of Lean 63 Description of System Components 63 Move Time 63 Wait Time 64 Setup Time 64 Process Time 64 Reducing or Eliminating Waste 64 Other Catego

In [61]:
# ============================================
# STEP 3.12 - Final Chunk Validation
# ============================================

print("============================================")
print("       IMPROVED CHUNKING VALIDATION")
print("============================================")

print("Total pages:", len(documents))
print("Total chunks:", len(chunks))

print("\nFirst chunk:")
print("ID:", chunks[0]["id"])
print("Source:", chunks[0]["source"])
print("Pages:", chunks[0]["start_page"], "→", chunks[0]["end_page"])
print("Words:", chunks[0]["word_count"])

print("\nLast chunk:")
print("ID:", chunks[-1]["id"])
print("Source:", chunks[-1]["source"])
print("Pages:", chunks[-1]["start_page"], "→", chunks[-1]["end_page"])
print("Words:", chunks[-1]["word_count"])

print("\n✅ IMPROVED CHUNKING COMPLETED!")

       IMPROVED CHUNKING VALIDATION
Total pages: 17
Total chunks: 8

First chunk:
ID: chunk_0
Source: 1.pdf
Pages: 1 → 4
Words: 500

Last chunk:
ID: chunk_7
Source: 1.pdf
Pages: 16 → 17
Words: 167

✅ IMPROVED CHUNKING COMPLETED!


**Loading Embedding Model**

In [62]:
# ============================================
# STEP 4.1 - Load Embedding Model
# ============================================

from sentence_transformers import SentenceTransformer

embedding_model_name = "BAAI/bge-small-en-v1.5"

embedding_model = SentenceTransformer(
    embedding_model_name
)

print("Embedding model loaded successfully! ✅")
print("Model:", embedding_model_name)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model loaded successfully! ✅
Model: BAAI/bge-small-en-v1.5


In [63]:
# ============================================
# STEP 4.2 - Test Embedding
# ============================================

test_text = chunks[0]["text"]

test_embedding = embedding_model.encode(
    test_text
)

print("Embedding generated successfully! ✅")
print("Embedding type:", type(test_embedding))
print("Embedding dimensions:", len(test_embedding))

Embedding generated successfully! ✅
Embedding type: <class 'numpy.ndarray'>
Embedding dimensions: 384


In [64]:
# ============================================
# STEP 4.3 - Generate Embeddings
# ============================================

chunk_texts = [
    chunk["text"]
    for chunk in chunks
]

embeddings = embedding_model.encode(
    chunk_texts,
    show_progress_bar=True
)

print("\nEmbeddings generated successfully! ✅")
print("Number of embeddings:", len(embeddings))
print("Embedding dimensions:", embeddings.shape[1])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Embeddings generated successfully! ✅
Number of embeddings: 8
Embedding dimensions: 384


**CHROMA DB**

In [65]:
# ============================================
# STEP 4.4 - Initialize ChromaDB
# ============================================

import chromadb

chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(
    name="research_documents"
)

print("ChromaDB collection created successfully! ✅")
print("Collection name:", collection.name)

ChromaDB collection created successfully! ✅
Collection name: research_documents


**STORE CHUNKS IN CHROMA DB**

In [66]:
# ============================================
# STEP 4.5 - Store Chunks in ChromaDB
# ============================================

ids = [
    chunk["id"]
    for chunk in chunks
]

documents_for_db = [
    chunk["text"]
    for chunk in chunks
]

metadatas = [
    {
        "source": chunk["source"],
        "start_page": chunk["start_page"],
        "end_page": chunk["end_page"]
    }
    for chunk in chunks
]

collection.add(
    ids=ids,
    documents=documents_for_db,
    embeddings=embeddings.tolist(),
    metadatas=metadatas
)

print("Chunks stored in ChromaDB successfully! ✅")
print("Total stored documents:", collection.count())

Chunks stored in ChromaDB successfully! ✅
Total stored documents: 8


**Test Semantic Search**

In [70]:
# ============================================
# STEP 4.6 - Semantic Search Test
# ============================================

query = "What is Lean Construction?"

query_embedding = embedding_model.encode(
    query
).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

print("Query:")
print(query)

print("\nRetrieved Results:")
print("=" * 80)

for i in range(len(results["documents"][0])):

    print(f"\nResult {i + 1}")

    print(
        "Pages:",
        results["metadatas"][0][i]["start_page"],
        "→",
        results["metadatas"][0][i]["end_page"]
    )

    print(
        "Text:",
        results["documents"][0][i][:700]
    )

Query:
What is Lean Construction?

Retrieved Results:

Result 1
Pages: 4 → 6
Text: Traditional Construction 59 Barriers to Applying Manufacturing Methods to Construction 60 Characteristics of Lean Construction 60 Lean Principles 61 Value 61 Value Stream 62 Value Stream Mapping 62 Flow 62 Pull 62 Perfection 63 Systems Perspective of Lean 63 Description of System Components 63 Move Time 63 Wait Time 64 Setup Time 64 Process Time 64 Reducing or Eliminating Waste 64 Other Categories of Waste 65 Lean Construction Fundamentals 66 Three Connected Opportunities 67 Five Big Ideas 67 Questions for Discussion 69 Appendix: ConsensusDocs 69 ConsensusDocs Endorsing Organizations 70 References 70 Bibliography 72 4 Lean Process Management 73 Operation of The Lean Project Delivery System 7

Result 2
Pages: 1 → 4
Text: Modern Construction Lean Project Delivery and Integrated Practices Lincoln H. Forbes Syed M. Ahmed CRC Press Taylor & Francis Croup Boca Raton London New York CRC Press is an imprint of t

In [69]:
# ============================================
# STEP 4.7 - Display Retrieved Sources
# ============================================

print("Query:", query)
print()

for i, metadata in enumerate(results["metadatas"][0]):

    print(
        f"Result {i + 1}: "
        f"{metadata['source']} | "
        f"Pages {metadata['start_page']}-{metadata['end_page']}"
    )

Query: What is Lean Construction?

Result 1: 1.pdf | Pages 4-6
Result 2: 1.pdf | Pages 1-4
Result 3: 1.pdf | Pages 16-17


In [71]:
# Step 5.1 — Install LLM library

!pip install -q google-generativeai

In [78]:
!pip install -q -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 25.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.44.0 which is incompatible.


In [79]:
from google import genai
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)

print("Gemini client configured successfully! ✅")

Gemini client configured successfully! ✅


In [85]:
response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="Explain Lean Construction in one simple sentence."
)

print("Gemini Response:")
print(response.text)

Gemini Response:
Lean construction is the practice of maximizing value and minimizing waste on a building project by improving efficiency, teamwork, and continuous flow from start to finish.


In [86]:
# Step 5.4 — Prepare retrieved context

retrieved_documents = results["documents"][0]
retrieved_metadatas = results["metadatas"][0]

context_parts = []

for i, (doc, metadata) in enumerate(
    zip(retrieved_documents, retrieved_metadatas),
    start=1
):

    context_parts.append(
        f"""
SOURCE {i}
Document: {metadata['source']}
Pages: {metadata['start_page']} - {metadata['end_page']}

Content:
{doc}
"""
    )

context = "\n".join(context_parts)

print("Retrieved context prepared successfully! ✅")
print("=" * 80)
print(context[:3000])

Retrieved context prepared successfully! ✅

SOURCE 1
Document: 1.pdf
Pages: 4 - 6

Content:
Traditional Construction 59 Barriers to Applying Manufacturing Methods to Construction 60 Characteristics of Lean Construction 60 Lean Principles 61 Value 61 Value Stream 62 Value Stream Mapping 62 Flow 62 Pull 62 Perfection 63 Systems Perspective of Lean 63 Description of System Components 63 Move Time 63 Wait Time 64 Setup Time 64 Process Time 64 Reducing or Eliminating Waste 64 Other Categories of Waste 65 Lean Construction Fundamentals 66 Three Connected Opportunities 67 Five Big Ideas 67 Questions for Discussion 69 Appendix: ConsensusDocs 69 ConsensusDocs Endorsing Organizations 70 References 70 Bibliography 72 4 Lean Process Management 73 Operation of The Lean Project Delivery System 73 Structure of The Lean Project Delivery System 74 Project Definition 75 Establishing Design Criteria 76 Lean Design Phase 76 Lean Supply 77 Lean Assembly 77 Production Control and Work Structuring 78 Lean De

In [87]:
# Step 5.5 — Basic RAG

query = "What is Lean Construction?"

prompt = f"""
You are an AI research document assistant.

Answer the user's question using ONLY the information
provided in the context below.

Do not use outside knowledge.
Do not make up information.

If the answer cannot be found in the context,
say: "I could not find this information in the provided document."

User Question:
{query}

Context:
{context}

Instructions:
1. Give a clear and concise answer.
2. Use only the provided document context.
3. Mention the relevant page numbers from the context.
"""

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=prompt
)

print("QUESTION:")
print(query)

print("\nANSWER:")
print(response.text)

QUESTION:
What is Lean Construction?

ANSWER:
Based on the provided documents, a specific definition of Lean Construction is outlined in Chapter 3 ("Foundations of Lean Construction"), which includes topics such as Lean Theory, origins, characteristics, and principles (such as value, value stream, flow, pull, and perfection). However, the exact text of the definition itself is located on page 45, which is listed in the table of contents under "Defining Lean Construction" (Document 2, Source 2, pages 1–4), but the text on page 45 is not included in the provided context excerpts. 

Therefore, I could not find the exact detailed definition of Lean Construction in the provided document context.


In [88]:
print("\nSOURCES:")
print("=" * 60)

for i, metadata in enumerate(retrieved_metadatas, start=1):
    print(
        f"{i}. {metadata['source']} | "
        f"Pages {metadata['start_page']}-{metadata['end_page']}"
    )


SOURCES:
1. 1.pdf | Pages 4-6
2. 1.pdf | Pages 1-4
3. 1.pdf | Pages 16-17


In [89]:
# Step 6.1 — Document Search Tool

def document_search(query, top_k=3):
    """
    Search the research documents using ChromaDB.
    Returns relevant document chunks with source information.
    """

    # Convert query into embedding
    query_embedding = embedding_model.encode(
        query
    ).tolist()

    # Search ChromaDB
    search_results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    retrieved_documents = search_results["documents"][0]
    retrieved_metadatas = search_results["metadatas"][0]

    results = []

    for doc, metadata in zip(
        retrieved_documents,
        retrieved_metadatas
    ):
        results.append({
            "text": doc,
            "source": metadata["source"],
            "start_page": metadata["start_page"],
            "end_page": metadata["end_page"]
        })

    return results

In [90]:
test_query = "What are the characteristics of Lean Construction?"

search_results = document_search(test_query)

print("SEARCH QUERY:")
print(test_query)

print("\nRETRIEVED DOCUMENTS:")
print("=" * 80)

for i, result in enumerate(search_results, start=1):

    print(f"\nResult {i}")
    print(
        f"Source: {result['source']} | "
        f"Pages: {result['start_page']}-{result['end_page']}"
    )

    print("Text:")
    print(result["text"][:500])

SEARCH QUERY:
What are the characteristics of Lean Construction?

RETRIEVED DOCUMENTS:

Result 1
Source: 1.pdf | Pages: 4-6
Text:
Traditional Construction 59 Barriers to Applying Manufacturing Methods to Construction 60 Characteristics of Lean Construction 60 Lean Principles 61 Value 61 Value Stream 62 Value Stream Mapping 62 Flow 62 Pull 62 Perfection 63 Systems Perspective of Lean 63 Description of System Components 63 Move Time 63 Wait Time 64 Setup Time 64 Process Time 64 Reducing or Eliminating Waste 64 Other Categories of Waste 65 Lean Construction Fundamentals 66 Three Connected Opportunities 67 Five Big Ideas 67 Qu

Result 2
Source: 1.pdf | Pages: 6-8
Text:
139 Sustaining Lean Initiatives 139 Qualities of a Lean Coach 140 Lean Coaching 140 Details of the Three As 142 Awareness 142 Acceptance 144 Action 144 Case of Ready Mechanical 144 Debrief of the Ready Mechanical Case 146 National Builder 146 Debrief of the National Builder Case 147 Examples of Lean Project Delivery Applicat

In [91]:
# Step 6.3 — Document Summarization Tool

def summarize_text(text):
    """
    Summarize provided document text using Gemini.
    """

    prompt = f"""
You are a research document assistant.

Summarize the following document content clearly and concisely.

Use ONLY the provided text.
Do not add outside information.

Document Content:
{text}

Provide:
1. Main idea
2. Important points
3. Key concepts
"""

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    return response.text

In [92]:
# Step 6.4 — Test summarization tool

sample_text = search_results[0]["text"]

summary = summarize_text(sample_text)

print("SUMMARY")
print("=" * 80)
print(summary)

SUMMARY
Based on the provided document, here is a clear and concise summary:

**1. Main idea**
The document outlines the principles, process management, measurement tools, and practical applications of lean construction, detailing how manufacturing methods, systems perspectives, and specific scheduling and analysis techniques are applied to improve construction projects.

**2. Important points**
*   **Lean Principles & Systems:** Covers traditional construction barriers, lean characteristics (value, value stream, flow, pull, perfection), system components, and waste reduction.
*   **Lean Process Management:** Details the Lean Project Delivery System—including project definition, lean design, supply, assembly, production control, set-based design, and target-value design—alongside the Last Planner® System (master, look-ahead, weekly, and daily work plans).
*   **Performance Measurement:** Focuses on measuring lean construction performance using statistical process control, reasons analy

In [93]:
# Step 6.5 — Calculator Tool

def calculator(expression):
    """
    Safely evaluate basic mathematical expressions.
    """

    allowed_chars = "0123456789+-*/().% "

    if not all(char in allowed_chars for char in expression):
        return "Invalid mathematical expression."

    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return result

    except Exception:
        return "Could not calculate the expression."

In [95]:
# Step 6.6 — Test calculator tool

test_expression = "250 * 0.15"

result = calculator(test_expression)

print("Expression:", test_expression)
print("Result:", result)

print(calculator("1000 / 4 + 50"))

Expression: 250 * 0.15
Result: 37.5
300.0


In [96]:
# Step 7.1 — LangGraph setup

from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
import operator

print("LangGraph imported successfully! ✅")

LangGraph imported successfully! ✅


In [97]:
# Step 7.2 — Define Agent State

class AgentState(TypedDict):
    query: str
    tool: str
    tool_result: str
    final_answer: str

print("Agent state created successfully! ✅")

Agent state created successfully! ✅


In [100]:
# Step 7.3 — Fixed Tool Selection

def select_tool(state: AgentState):

    query = state["query"].lower()

    # Calculator keywords and mathematical symbols
    calculator_keywords = [
        "calculate",
        "percentage",
        "percent",
        "multiply",
        "divide",
        "plus",
        "minus",
        "times",
        "average"
    ]

    has_math_symbol = any(
        symbol in query
        for symbol in ["%", "*", "/", "+", "-"]
    )

    has_calculator_keyword = any(
        word in query
        for word in calculator_keywords
    )

    if has_math_symbol or has_calculator_keyword:
        tool = "calculator"

    elif any(word in query for word in [
        "summarize",
        "summary",
        "summarise",
        "give me a summary"
    ]):
        tool = "summarizer"

    else:
        tool = "document_search"

    return {
        "tool": tool
    }

In [101]:
# Test fixed tool selection

test_questions = [
    "What is Lean Construction?",
    "Summarize Lean Construction",
    "What is 25% of 800?",
    "Calculate 250 * 0.15",
    "What is 1000 / 4?"
]

for question in test_questions:

    state = {
        "query": question,
        "tool": "",
        "tool_result": "",
        "final_answer": ""
    }

    result = select_tool(state)

    print("Question:", question)
    print("Selected Tool:", result["tool"])
    print("-" * 60)

Question: What is Lean Construction?
Selected Tool: document_search
------------------------------------------------------------
Question: Summarize Lean Construction
Selected Tool: summarizer
------------------------------------------------------------
Question: What is 25% of 800?
Selected Tool: calculator
------------------------------------------------------------
Question: Calculate 250 * 0.15
Selected Tool: calculator
------------------------------------------------------------
Question: What is 1000 / 4?
Selected Tool: calculator
------------------------------------------------------------


In [102]:
# Step 7.5 — Tool Execution Nodes

def document_search_node(state: AgentState):

    results = document_search(state["query"])

    formatted_results = []

    for result in results:
        formatted_results.append(
            f"""
Source: {result['source']}
Pages: {result['start_page']}-{result['end_page']}

Content:
{result['text']}
"""
        )

    tool_result = "\n".join(formatted_results)

    return {
        "tool_result": tool_result
    }


def summarizer_node(state: AgentState):

    # First search relevant content
    results = document_search(state["query"])

    combined_text = "\n\n".join(
        result["text"]
        for result in results
    )

    summary = summarize_text(combined_text)

    return {
        "tool_result": summary
    }


def calculator_node(state: AgentState):

    query = state["query"]

    # Extract simple mathematical expression
    expression = query

    # Convert percentage questions like:
    # "What is 25% of 800?"
    match = re.search(
        r"(\d+(?:\.\d+)?)\s*%\s*of\s*(\d+(?:\.\d+)?)",
        query
    )

    if match:

        percentage = float(match.group(1))
        number = float(match.group(2))

        result = (percentage / 100) * number

        tool_result = str(result)

    else:

        # Try to extract mathematical expression
        expression = re.sub(
            r"[^0-9+\-*/().% ]",
            "",
            query
        )

        tool_result = str(
            calculator(expression)
        )

    return {
        "tool_result": tool_result
    }

print("Tool nodes created successfully! ✅")

Tool nodes created successfully! ✅


In [103]:
test_state = {
    "query": "What is Lean Construction?",
    "tool": "document_search",
    "tool_result": "",
    "final_answer": ""
}

result = document_search_node(test_state)

print(result["tool_result"][:1500])


Source: 1.pdf
Pages: 4-6

Content:
Traditional Construction 59 Barriers to Applying Manufacturing Methods to Construction 60 Characteristics of Lean Construction 60 Lean Principles 61 Value 61 Value Stream 62 Value Stream Mapping 62 Flow 62 Pull 62 Perfection 63 Systems Perspective of Lean 63 Description of System Components 63 Move Time 63 Wait Time 64 Setup Time 64 Process Time 64 Reducing or Eliminating Waste 64 Other Categories of Waste 65 Lean Construction Fundamentals 66 Three Connected Opportunities 67 Five Big Ideas 67 Questions for Discussion 69 Appendix: ConsensusDocs 69 ConsensusDocs Endorsing Organizations 70 References 70 Bibliography 72 4 Lean Process Management 73 Operation of The Lean Project Delivery System 73 Structure of The Lean Project Delivery System 74 Project Definition 75 Establishing Design Criteria 76 Lean Design Phase 76 Lean Supply 77 Lean Assembly 77 Production Control and Work Structuring 78 Lean Design Details 78 Lean Design Management 80 Designing for 

In [104]:
test_state = {
    "query": "What is 25% of 800?",
    "tool": "calculator",
    "tool_result": "",
    "final_answer": ""
}

result = calculator_node(test_state)

print("Calculator Result:", result["tool_result"])

Calculator Result: 200.0


In [105]:
# Step 7.7 — Build LangGraph Workflow

def route_to_tool(state: AgentState):
    return state["tool"]


def run_selected_tool(state: AgentState):

    tool = state["tool"]

    if tool == "document_search":
        return document_search_node(state)

    elif tool == "summarizer":
        return summarizer_node(state)

    elif tool == "calculator":
        return calculator_node(state)

    else:
        return {
            "tool_result": "Unknown tool."
        }


# Create graph
graph = StateGraph(AgentState)

# Add nodes
graph.add_node("select_tool", select_tool)
graph.add_node("run_tool", run_selected_tool)

# Start → tool selection
graph.add_edge(START, "select_tool")

# Tool selection → tool execution
graph.add_edge("select_tool", "run_tool")

# Tool execution → end
graph.add_edge("run_tool", END)

# Compile graph
agent = graph.compile()

print("LangGraph Agent created successfully! ✅")

LangGraph Agent created successfully! ✅


In [106]:
# Step 7.8 — Test Agent

initial_state = {
    "query": "What is Lean Construction?",
    "tool": "",
    "tool_result": "",
    "final_answer": ""
}

result = agent.invoke(initial_state)

print("USER QUERY:")
print(result["query"])

print("\nSELECTED TOOL:")
print(result["tool"])

print("\nTOOL RESULT:")
print(result["tool_result"][:2000])

USER QUERY:
What is Lean Construction?

SELECTED TOOL:
document_search

TOOL RESULT:

Source: 1.pdf
Pages: 4-6

Content:
Traditional Construction 59 Barriers to Applying Manufacturing Methods to Construction 60 Characteristics of Lean Construction 60 Lean Principles 61 Value 61 Value Stream 62 Value Stream Mapping 62 Flow 62 Pull 62 Perfection 63 Systems Perspective of Lean 63 Description of System Components 63 Move Time 63 Wait Time 64 Setup Time 64 Process Time 64 Reducing or Eliminating Waste 64 Other Categories of Waste 65 Lean Construction Fundamentals 66 Three Connected Opportunities 67 Five Big Ideas 67 Questions for Discussion 69 Appendix: ConsensusDocs 69 ConsensusDocs Endorsing Organizations 70 References 70 Bibliography 72 4 Lean Process Management 73 Operation of The Lean Project Delivery System 73 Structure of The Lean Project Delivery System 74 Project Definition 75 Establishing Design Criteria 76 Lean Design Phase 76 Lean Supply 77 Lean Assembly 77 Production Control an

In [107]:
# Step 7.9 — Gemini Final Answer Node

def final_answer_node(state: AgentState):

    query = state["query"]
    tool = state["tool"]
    tool_result = state["tool_result"]

    prompt = f"""
You are an AI Research Document Assistant.

User Question:
{query}

Tool Used:
{tool}

Tool Result:
{tool_result}

Instructions:

1. Answer the user's question using the tool result.
2. If the tool result comes from the research document,
   use only information found in that document.
3. Do not invent information.
4. Give a clear and concise answer.
5. If document sources and page numbers are available,
   mention them in the answer.
6. For calculations, clearly provide the calculated result.
"""

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    return {
        "final_answer": response.text
    }

print("Final answer node created successfully! ✅")

Final answer node created successfully! ✅


In [108]:
# Step 7.10 — Add Final Answer Node to Graph

graph = StateGraph(AgentState)

# Add nodes
graph.add_node("select_tool", select_tool)
graph.add_node("run_tool", run_selected_tool)
graph.add_node("final_answer", final_answer_node)

# Workflow
graph.add_edge(START, "select_tool")
graph.add_edge("select_tool", "run_tool")
graph.add_edge("run_tool", "final_answer")
graph.add_edge("final_answer", END)

# Compile
agent = graph.compile()

print("Updated LangGraph Agent created successfully! ✅")

Updated LangGraph Agent created successfully! ✅


In [109]:
# Step 7.11 — Test Complete Agent

initial_state = {
    "query": "What is Lean Construction?",
    "tool": "",
    "tool_result": "",
    "final_answer": ""
}

result = agent.invoke(initial_state)

print("USER QUESTION:")
print(result["query"])

print("\nSELECTED TOOL:")
print(result["tool"])

print("\nFINAL ANSWER:")
print(result["final_answer"])

USER QUESTION:
What is Lean Construction?

SELECTED TOOL:
document_search

FINAL ANSWER:
Based on the provided research document, Lean Construction is a production management-based approach to project delivery that adapts manufacturing methods (such as those from the Toyota Production System) to construction. Its foundations and principles include:

* **Core Lean Principles:** Value, Value Stream, Value Stream Mapping, Flow, Pull, and Perfection (Source: 1.pdf, Pages 4-6).
* **Key Components & Systems:** It incorporates a systems perspective (evaluating move time, wait time, setup time, and process time), focuses on reducing or eliminating waste, and relies on frameworks like the Lean Project Delivery System, Target-Value Design, Set-Based Design, and the Last Planner® System for production control (Source: 1.pdf, Pages 4-6).
* **Philosophical Difference:** It contrasts with traditional construction methods by shifting away from traditional planning, scheduling, and push systems toward

In [110]:
# Step 8.1 — Add conversation memory to Agent State

from typing import TypedDict

class AgentState(TypedDict):
    query: str
    tool: str
    tool_result: str
    final_answer: str
    conversation_history: list

print("Agent state with memory created successfully! ✅")

Agent state with memory created successfully! ✅


In [111]:
# Step 8.2 — Test conversation memory

conversation_history = []

conversation_history.append({
    "user": "What is Lean Construction?",
    "assistant": "Lean Construction is a production management-based approach to construction."
})

conversation_history.append({
    "user": "What are its main principles?",
    "assistant": "The main principles include Value, Value Stream, Flow, Pull, and Perfection."
})

print("Conversation History:")
print("=" * 60)

for message in conversation_history:
    print("User:", message["user"])
    print("Assistant:", message["assistant"])
    print()

Conversation History:
User: What is Lean Construction?
Assistant: Lean Construction is a production management-based approach to construction.

User: What are its main principles?
Assistant: The main principles include Value, Value Stream, Flow, Pull, and Perfection.



In [112]:
# Step 8.3 — Memory-aware Final Answer Node

def final_answer_node(state: AgentState):

    query = state["query"]
    tool = state["tool"]
    tool_result = state["tool_result"]
    history = state.get("conversation_history", [])

    history_text = ""

    for message in history:
        history_text += (
            f"User: {message['user']}\n"
            f"Assistant: {message['assistant']}\n\n"
        )

    prompt = f"""
You are an AI Research Document Assistant.

Previous Conversation:
{history_text}

Current User Question:
{query}

Tool Used:
{tool}

Tool Result:
{tool_result}

Instructions:

1. Use the previous conversation to understand follow-up questions.
2. Use the tool result to answer the current question.
3. For document questions, use only information from the provided document.
4. Do not invent information.
5. Give a clear and concise answer.
6. Mention source pages when available.
"""

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    return {
        "final_answer": response.text
    }

print("Memory-aware final answer node created! ✅")

Memory-aware final answer node created! ✅


In [113]:
graph = StateGraph(AgentState)

graph.add_node("select_tool", select_tool)
graph.add_node("run_tool", run_selected_tool)
graph.add_node("final_answer", final_answer_node)

graph.add_edge(START, "select_tool")
graph.add_edge("select_tool", "run_tool")
graph.add_edge("run_tool", "final_answer")
graph.add_edge("final_answer", END)

agent = graph.compile()

print("Memory-aware LangGraph agent compiled successfully! ✅")

Memory-aware LangGraph agent compiled successfully! ✅


In [114]:
conversation_history = []

def chat_with_agent(query, conversation_history):

    initial_state = {
        "query": query,
        "tool": "",
        "tool_result": "",
        "final_answer": "",
        "conversation_history": conversation_history
    }

    result = agent.invoke(initial_state)

    # Save current conversation into memory
    conversation_history.append({
        "user": query,
        "assistant": result["final_answer"]
    })

    return result["final_answer"], result

In [116]:
answer1, result1 = chat_with_agent(
    "What is Lean Construction?",
    conversation_history
)

print("Assistant:")
print(answer1)

print("\n" + "=" * 70 + "\n")

answer2, result2 = chat_with_agent(
    "What are its main principles?",
    conversation_history
)

print("Assistant:")
print(answer2)
print(conversation_history)

Assistant:
Based on the provided documents, Lean Construction is a production management-based approach to project delivery that adapts manufacturing methods—specifically drawing from the Toyota Production System—to the construction industry (Source 1, p. 3 / page 45). 

It focuses on:
* **Core Principles:** Defining value, mapping the value stream, ensuring continuous flow, utilizing pull systems, and striving for perfection (Source 1, p. 4 / pages 61–64).
* **Systems Perspective:** Optimizing the overall system by analyzing and reducing components such as move time, wait time, setup time, and process time, while eliminating waste (Source 1, p. 4 / pages 63–64).
* **Process Improvement:** Differing fundamentally from traditional construction methods by using modern production control and management tools—such as the Last Planner® System, target-value design, and work structuring—to improve performance and productivity (Source 1, p. 3–4 / pages 45, 59–60, 83, 86).

*(Source: 1.pdf, Pag

In [117]:
def llm_select_tool(query):

    prompt = f"""
You are a tool-selection system for an AI Research Document Assistant.

Available tools:

1. document_search
   - Use for questions about the research document.
   - Examples:
     "What is Lean Construction?"
     "What are the main principles?"
     "Explain value stream."

2. summarizer
   - Use when the user explicitly asks for a summary.
   - Examples:
     "Summarize Lean Construction."
     "Give me a summary of this document."

3. calculator
   - Use for mathematical calculations.
   - Examples:
     "What is 25% of 800?"
     "Calculate 250 * 0.15."

User Query:
{query}

Return ONLY one of these exact tool names:

document_search
summarizer
calculator
"""

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    selected_tool = response.text.strip().lower()

    if selected_tool not in [
        "document_search",
        "summarizer",
        "calculator"
    ]:
        selected_tool = "document_search"

    return selected_tool

In [118]:
test_queries = [
    "What is Lean Construction?",
    "What are its main principles?",
    "Summarize Lean Construction",
    "What is 25% of 800?",
    "Calculate 250 * 0.15"
]

for query in test_queries:
    tool = llm_select_tool(query)

    print("Query:", query)
    print("Selected Tool:", tool)
    print("-" * 60)

Query: What is Lean Construction?
Selected Tool: document_search
------------------------------------------------------------
Query: What are its main principles?
Selected Tool: document_search
------------------------------------------------------------
Query: Summarize Lean Construction
Selected Tool: summarizer
------------------------------------------------------------
Query: What is 25% of 800?
Selected Tool: calculator
------------------------------------------------------------
Query: Calculate 250 * 0.15
Selected Tool: calculator
------------------------------------------------------------


In [119]:
def select_tool(state: AgentState):

    query = state["query"]

    tool = llm_select_tool(query)

    return {
        "tool": tool
    }

In [120]:
graph = StateGraph(AgentState)

graph.add_node("select_tool", select_tool)
graph.add_node("run_tool", run_selected_tool)
graph.add_node("final_answer", final_answer_node)

graph.add_edge(START, "select_tool")
graph.add_edge("select_tool", "run_tool")
graph.add_edge("run_tool", "final_answer")
graph.add_edge("final_answer", END)

agent = graph.compile()

print("LLM-based Agent compiled successfully! ✅")

LLM-based Agent compiled successfully! ✅


In [121]:
conversation_history = []

answer, result = chat_with_agent(
    "What is Lean Construction?",
    conversation_history
)

print("Selected Tool:", result["tool"])
print("\nFinal Answer:")
print(answer)

Selected Tool: document_search

Final Answer:
Based on the provided document, **Lean Construction** is an approach that applies manufacturing methods and lean principles (such as defining value, mapping the value stream, ensuring flow, establishing pull, and striving for perfection) to the construction industry (Source: 1.pdf, pages 4-6). It represents a shift from traditional construction methods, addressing productivity issues by focusing on system perspectives, reducing or eliminating waste, and utilizing structured processes like the Lean Project Delivery System (Source: 1.pdf, pages 4-6). 

*(Source: 1.pdf, Pages 4–6)*


In [122]:
answer, result = chat_with_agent(
    "What is 25% of 800?",
    conversation_history
)

print("Selected Tool:", result["tool"])
print("\nFinal Answer:")
print(answer)

Selected Tool: calculator

Final Answer:
25% of 800 is 200.


In [123]:
conversation_history = []

test_queries = [
    "What is Lean Construction?",
    "Summarize Lean Construction.",
    "What is 25% of 800?"
]

for query in test_queries:

    answer, result = chat_with_agent(
        query,
        conversation_history
    )

    print("=" * 70)
    print("User:", query)
    print("Selected Tool:", result["tool"])
    print("\nAssistant:")
    print(answer)
    print()

User: What is Lean Construction?
Selected Tool: document_search

Assistant:
Based on the provided documents, Lean Construction is a production management-based approach to project delivery that adapts manufacturing methods (such as those from the Toyota Production System) to the construction industry. It contrasts with traditional construction methods and focuses on specific principles:

* **Core Principles:** Value, Value Stream, Flow, Pull, and Perfection (Source 1, pages 4–6).
* **Key Characteristics:** It involves a systems perspective of lean, reducing or eliminating waste (including move time, wait time, setup time, and process time), and utilizing tools and frameworks like the Lean Project Delivery System, Target-Value Design, Set-Based Design, and the Last Planner® System (Source 1, pages 4–6).

*(Source: 1.pdf, Pages 1–6)*

User: Summarize Lean Construction.
Selected Tool: summarizer

Assistant:
Based on the provided documents, here is a summary of Lean Construction:

* **Main

In [124]:
def document_search_node(state: AgentState):

    results = document_search(state["query"])

    formatted_results = []

    for i, result in enumerate(results, start=1):

        formatted_results.append({
            "source": result["source"],
            "start_page": result["start_page"],
            "end_page": result["end_page"],
            "text": result["text"]
        })

    return {
        "tool_result": json.dumps(
            formatted_results,
            indent=2
        )
    }

In [125]:
def summarizer_node(state: AgentState):

    results = document_search(state["query"])

    combined_text = "\n\n".join(
        result["text"]
        for result in results
    )

    summary = summarize_text(combined_text)

    sources = []

    for result in results:
        sources.append({
            "source": result["source"],
            "start_page": result["start_page"],
            "end_page": result["end_page"]
        })

    tool_output = {
        "summary": summary,
        "sources": sources
    }

    return {
        "tool_result": json.dumps(
            tool_output,
            indent=2
        )
    }

In [127]:
def calculator_node(state: AgentState):

    query = state["query"]

    match = re.search(
        r"(\d+(?:\.\d+)?)\s*%\s*of\s*(\d+(?:\.\d+)?)",
        query
    )

    if match:

        percentage = float(match.group(1))
        number = float(match.group(2))

        result = (percentage / 100) * number

    else:

        expression = re.sub(
            r"[^0-9+\-*/().% ]",
            "",
            query
        )

        result = calculator(expression)

    tool_output = {
        "expression": query,
        "result": result
    }

    return {
        "tool_result": json.dumps(
            tool_output,
            indent=2
        )
    }

In [128]:
def run_selected_tool(state: AgentState):

    tool = state["tool"]

    if tool == "document_search":
        return document_search_node(state)

    elif tool == "summarizer":
        return summarizer_node(state)

    elif tool == "calculator":
        return calculator_node(state)

    else:
        return {
            "tool_result": json.dumps({
                "error": "Unknown tool selected."
            })
        }


print("Updated tool runner created successfully! ✅")

Updated tool runner created successfully! ✅


In [129]:
graph = StateGraph(AgentState)

# Nodes
graph.add_node("select_tool", select_tool)
graph.add_node("run_tool", run_selected_tool)
graph.add_node("final_answer", final_answer_node)

# Flow
graph.add_edge(START, "select_tool")
graph.add_edge("select_tool", "run_tool")
graph.add_edge("run_tool", "final_answer")
graph.add_edge("final_answer", END)

# Compile
agent = graph.compile()

print("Updated Agentic RAG graph compiled successfully! ✅")

Updated Agentic RAG graph compiled successfully! ✅


In [130]:
conversation_history = []

answer, result = chat_with_agent(
    "What is Lean Construction?",
    conversation_history
)

print("Selected Tool:", result["tool"])
print("\nFinal Answer:")
print(answer)

Selected Tool: document_search

Final Answer:
Based on the provided documents, **Lean Construction** is an approach to construction management that applies manufacturing methods and principles (originating largely from the Toyota Production System) to improve project performance, productivity, and delivery compared to traditional construction methods (Source 1.pdf, pages 1-4, 45). 

Key aspects of Lean Construction include:
* **Core Principles:** Focusing on value, the value stream, flow, pull, and perfection (Source 1.pdf, page 4).
* **System Components:** Managing and reducing components like move time, wait time, setup time, and process time, while eliminating waste (Source 1.pdf, page 4).
* **Lean Project Delivery:** Utilizing structured phases—such as project definition, lean design, lean supply, lean assembly, and production control (like the Last Planner® System)—to maximize value and minimize waste across the project lifecycle (Source 1.pdf, pages 4, 16).


In [131]:
conversation_history = []

test_queries = [
    "What is Lean Construction?",
    "Summarize Lean Construction.",
    "What is 25% of 800?"
]

for query in test_queries:

    answer, result = chat_with_agent(
        query,
        conversation_history
    )

    print("=" * 80)
    print("USER:", query)
    print("SELECTED TOOL:", result["tool"])
    print("\nASSISTANT:")
    print(answer)
    print()

USER: What is Lean Construction?
SELECTED TOOL: document_search

ASSISTANT:
Based on the provided document, Chapter 3 is dedicated to the "Foundations of Lean Construction" and covers topics such as lean theory, the origins of lean construction, lean design and construction, differences between lean and traditional construction, and characteristics and principles of lean construction (such as value, value stream, flow, pull, and perfection) (Source 1.pdf, pages 1-4). 

However, the specific text defining "Lean Construction" appears in Chapter 3 on page 45 (which falls within the text range of source pages 1–4 of the document, specifically listed in the Table of Contents on page xi). A concise definition is rooted in applying manufacturing-based lean principles—such as the Toyota Production System (Just-In-Time concepts)—to construction processes to eliminate waste, improve flow, maximize value, and overcome inefficiencies found in traditional construction methods (Source 1.pdf, pages 1

**EVALUATION**

In [133]:
evaluation_questions = [
    {
        "question": "What is Lean Construction?",
        "expected_topic": "Lean Construction definition"
    },
    {
        "question": "What are the main principles of Lean Construction?",
        "expected_topic": "Value, Value Stream, Flow, Pull, Perfection"
    },
    {
        "question": "What is the purpose of Lean Construction?",
        "expected_topic": "Reducing waste and improving project performance"
    },
    {
        "question": "What is the Last Planner System?",
        "expected_topic": "Production planning and control"
    },
    {
        "question": "What is Value Stream Mapping?",
        "expected_topic": "Mapping processes and identifying waste"
    }
]

print("Evaluation dataset created successfully! ✅")
print("Number of questions:", len(evaluation_questions))

Evaluation dataset created successfully! ✅
Number of questions: 5


In [134]:
for item in evaluation_questions:

    question = item["question"]

    results = document_search(
        question,
        top_k=3
    )

    print("=" * 80)
    print("Question:", question)
    print("Expected Topic:", item["expected_topic"])

    print("\nRetrieved Sources:")

    for i, result in enumerate(results, start=1):

        print(
            f"{i}. {result['source']} | "
            f"Pages {result['start_page']}-{result['end_page']}"
        )

    print()

Question: What is Lean Construction?
Expected Topic: Lean Construction definition

Retrieved Sources:
1. 1.pdf | Pages 4-6
2. 1.pdf | Pages 1-4
3. 1.pdf | Pages 16-17

Question: What are the main principles of Lean Construction?
Expected Topic: Value, Value Stream, Flow, Pull, Perfection

Retrieved Sources:
1. 1.pdf | Pages 4-6
2. 1.pdf | Pages 16-17
3. 1.pdf | Pages 1-4

Question: What is the purpose of Lean Construction?
Expected Topic: Reducing waste and improving project performance

Retrieved Sources:
1. 1.pdf | Pages 4-6
2. 1.pdf | Pages 1-4
3. 1.pdf | Pages 16-17

Question: What is the Last Planner System?
Expected Topic: Production planning and control

Retrieved Sources:
1. 1.pdf | Pages 4-6
2. 1.pdf | Pages 16-17
3. 1.pdf | Pages 8-10

Question: What is Value Stream Mapping?
Expected Topic: Mapping processes and identifying waste

Retrieved Sources:
1. 1.pdf | Pages 4-6
2. 1.pdf | Pages 16-17
3. 1.pdf | Pages 6-8



In [135]:
id="e8k4qp"
evaluation_keywords = {
    "Lean Construction definition": [
        "lean construction",
        "production management",
        "toyota production system"
    ],

    "Value, Value Stream, Flow, Pull, Perfection": [
        "value",
        "value stream",
        "flow",
        "pull",
        "perfection"
    ],

    "Reducing waste and improving project performance": [
        "waste",
        "productivity",
        "performance"
    ],

    "Production planning and control": [
        "last planner",
        "planning",
        "production control"
    ],

    "Mapping processes and identifying waste": [
        "value stream mapping",
        "value stream",
        "waste"
    ]
}


def calculate_relevance_score(question, expected_topic, top_k=3):

    results = document_search(
        question,
        top_k=top_k
    )

    keywords = evaluation_keywords[expected_topic]

    relevant_chunks = 0

    for result in results:

        text = result["text"].lower()

        if any(keyword.lower() in text for keyword in keywords):
            relevant_chunks += 1

    score = relevant_chunks / len(results)

    return score, relevant_chunks, len(results)

In [136]:
id="p6m2rx"
evaluation_results = []

for item in evaluation_questions:

    score, relevant, total = calculate_relevance_score(
        item["question"],
        item["expected_topic"]
    )

    evaluation_results.append({
        "question": item["question"],
        "score": score,
        "relevant_chunks": relevant,
        "total_chunks": total
    })

    print("=" * 70)
    print("Question:", item["question"])
    print("Relevant chunks:", relevant, "/", total)
    print("Relevance score:", round(score, 2))

Question: What is Lean Construction?
Relevant chunks: 3 / 3
Relevance score: 1.0
Question: What are the main principles of Lean Construction?
Relevant chunks: 2 / 3
Relevance score: 0.67
Question: What is the purpose of Lean Construction?
Relevant chunks: 2 / 3
Relevance score: 0.67
Question: What is the Last Planner System?
Relevant chunks: 3 / 3
Relevance score: 1.0
Question: What is Value Stream Mapping?
Relevant chunks: 1 / 3
Relevance score: 0.33


In [137]:
scores = [
    result["score"]
    for result in evaluation_results
]

average_score = sum(scores) / len(scores)

print("Average Retrieval Relevance Score:")
print(round(average_score, 2))

Average Retrieval Relevance Score:
0.73


In [138]:
ground_truth_keywords = {
    "Lean Construction definition": [
        "lean construction",
        "production management",
        "toyota production system"
    ],

    "Value, Value Stream, Flow, Pull, Perfection": [
        "value",
        "value stream",
        "flow",
        "pull",
        "perfection"
    ],

    "Reducing waste and improving project performance": [
        "waste",
        "productivity",
        "performance"
    ],

    "Production planning and control": [
        "last planner",
        "production control",
        "planning"
    ],

    "Mapping processes and identifying waste": [
        "value stream mapping",
        "value stream",
        "waste"
    ]
}


def evaluate_context(question, expected_topic, top_k=3):

    results = document_search(
        question,
        top_k=top_k
    )

    keywords = ground_truth_keywords[expected_topic]

    # Combine retrieved chunks
    retrieved_context = " ".join(
        result["text"].lower()
        for result in results
    )

    # Find expected keywords present in retrieved context
    found_keywords = []

    for keyword in keywords:
        if keyword.lower() in retrieved_context:
            found_keywords.append(keyword)

    # Context Recall
    context_recall = (
        len(found_keywords) / len(keywords)
        if keywords
        else 0
    )

    # Context Precision
    relevant_chunks = 0

    for result in results:

        text = result["text"].lower()

        if any(
            keyword.lower() in text
            for keyword in keywords
        ):
            relevant_chunks += 1

    context_precision = (
        relevant_chunks / len(results)
        if results
        else 0
    )

    return {
        "context_precision": context_precision,
        "context_recall": context_recall,
        "found_keywords": found_keywords,
        "total_keywords": len(keywords),
        "relevant_chunks": relevant_chunks,
        "total_chunks": len(results)
    }

In [139]:
context_evaluation = []

for item in evaluation_questions:

    result = evaluate_context(
        item["question"],
        item["expected_topic"]
    )

    context_evaluation.append({
        "question": item["question"],
        "context_precision": result["context_precision"],
        "context_recall": result["context_recall"]
    })

    print("=" * 70)
    print("Question:", item["question"])

    print(
        "Context Precision:",
        round(result["context_precision"], 2)
    )

    print(
        "Context Recall:",
        round(result["context_recall"], 2)
    )

    print(
        "Found Keywords:",
        result["found_keywords"]
    )

Question: What is Lean Construction?
Context Precision: 1.0
Context Recall: 0.33
Found Keywords: ['lean construction']
Question: What are the main principles of Lean Construction?
Context Precision: 0.67
Context Recall: 1.0
Found Keywords: ['value', 'value stream', 'flow', 'pull', 'perfection']
Question: What is the purpose of Lean Construction?
Context Precision: 0.67
Context Recall: 1.0
Found Keywords: ['waste', 'productivity', 'performance']
Question: What is the Last Planner System?
Context Precision: 1.0
Context Recall: 1.0
Found Keywords: ['last planner', 'production control', 'planning']
Question: What is Value Stream Mapping?
Context Precision: 0.33
Context Recall: 1.0
Found Keywords: ['value stream mapping', 'value stream', 'waste']


In [140]:
average_precision = sum(
    item["context_precision"]
    for item in context_evaluation
) / len(context_evaluation)

average_recall = sum(
    item["context_recall"]
    for item in context_evaluation
) / len(context_evaluation)

print("Average Context Precision:",
      round(average_precision, 2))

print("Average Context Recall:",
      round(average_recall, 2))

Average Context Precision: 0.73
Average Context Recall: 0.87


In [141]:
def evaluate_faithfulness(question):

    # Retrieve relevant context
    results = document_search(
        question,
        top_k=3
    )

    context = "\n\n".join(
        result["text"]
        for result in results
    )

    # Generate the actual RAG answer
    answer, agent_result = chat_with_agent(
        question,
        []
    )

    prompt = f"""
You are evaluating the faithfulness of an AI research assistant.

Question:
{question}

Retrieved Context:
{context}

Assistant Answer:
{answer}

Task:

Determine whether the assistant answer is supported by
the retrieved context.

Return ONLY valid JSON in this format:

{{
    "faithfulness_score": 0.0,
    "explanation": "short explanation"
}}

Scoring:

1.0 = Answer is fully supported by the context.
0.5 = Answer is partially supported.
0.0 = Answer is not supported by the context.

Do not use outside knowledge.
"""

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    return answer, response.text

In [142]:
answer, evaluation = evaluate_faithfulness(
    "What is Lean Construction?"
)

print("ASSISTANT ANSWER:")
print(answer)

print("\n" + "=" * 70)

print("FAITHFULNESS EVALUATION:")
print(evaluation)

ASSISTANT ANSWER:
Based on the provided document, Chapter 3 is dedicated to the "Foundations of Lean Construction" and covers topics such as defining lean construction, lean theory, the origins and adoption of relational contracting, characteristics of lean construction, lean principles (value, value stream, flow, pull, and perfection), and a systems perspective of lean (source: *1.pdf*, pages 1-4). 

*(Note: The document's table of contents indicates that Lean Construction involves applying manufacturing methods and principles—like reducing waste and improving flow—to the construction process, differing philosophically from traditional construction methods [sources: 1.pdf, pages 4–6].)*

FAITHFULNESS EVALUATION:
{
    "faithfulness_score": 1.0,
    "explanation": "The assistant's answer accurately cites information directly from the provided table of contents, such as Chapter 3 covering the foundations of lean construction, lean principles (value, value stream, flow, pull, perfection)

In [143]:
def evaluate_answer_relevancy(question):

    answer, agent_result = chat_with_agent(
        question,
        []
    )

    prompt = f"""
You are evaluating the relevance of an AI assistant's answer.

User Question:
{question}

Assistant Answer:
{answer}

Evaluate whether the answer directly addresses the user's question.

Return ONLY valid JSON:

{{
    "relevancy_score": 0.0,
    "explanation": "short explanation"
}}

Scoring:

1.0 = Directly and completely answers the question.
0.5 = Partially answers the question or contains unnecessary information.
0.0 = Does not answer the question.

Do not evaluate factual correctness.
Only evaluate whether the answer is relevant to the question.
"""

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    return answer, response.text

In [144]:
answer, evaluation = evaluate_answer_relevancy(
    "What are the main principles of Lean Construction?"
)

print("ASSISTANT ANSWER:")
print(answer)

print("\n" + "=" * 70)

print("ANSWER RELEVANCY EVALUATION:")
print(evaluation)

ASSISTANT ANSWER:
Based on the provided document, the main principles of Lean Construction (found on pages 61–63, source `1.pdf`) are:

* **Value:** Defining value from the perspective of the customer/client.
* **Value Stream:** Identifying and mapping the entire stream of processes required to deliver that value.
* **Flow:** Ensuring that the processes flow smoothly without interruptions, bottlenecks, or waste.
* **Pull:** Producing work only when it is needed by the downstream process (pull-driven scheduling rather than push).
* **Perfection:** Continuously striving for continuous improvement and perfection by relentlessly removing waste.

ANSWER RELEVANCY EVALUATION:
```json
{
    "relevancy_score": 1.0,
    "explanation": "The assistant's answer directly and completely addresses the user's question by listing the main principles of Lean Construction."
}
```


In [147]:
def run_light_evaluation():

    report = []

    for item in evaluation_questions:

        question = item["question"]
        expected_topic = item["expected_topic"]

        # Retrieval + context evaluation
        retrieval_score, relevant_chunks, total_chunks = (
            calculate_relevance_score(
                question,
                expected_topic
            )
        )

        context_result = evaluate_context(
            question,
            expected_topic
        )

        # Generate answer
        answer, agent_result = chat_with_agent(
            question,
            []
        )

        # One Gemini call for both
        context = "\n\n".join(
            result["text"]
            for result in document_search(
                question,
                top_k=3
            )
        )

        evaluation_prompt = f"""
You are evaluating an AI research assistant.

Question:
{question}

Retrieved Context:
{context}

Assistant Answer:
{answer}

Evaluate TWO things:

1. Faithfulness:
Is the answer supported by the retrieved context?

2. Answer Relevancy:
Does the answer directly address the question?

Return ONLY valid JSON:

{{
    "faithfulness": 0.0,
    "relevancy": 0.0
}}

Scoring:
1.0 = excellent
0.5 = partial
0.0 = poor
"""

        response = client.models.generate_content(
            model="gemini-3.5-flash-lite",
            contents=evaluation_prompt
        )

        try:
            evaluation = json.loads(response.text)

        except:
            evaluation = {
                "faithfulness": 0.0,
                "relevancy": 0.0
            }

        report.append({
            "question": question,
            "retrieval_relevance": retrieval_score,
            "context_precision": context_result[
                "context_precision"
            ],
            "context_recall": context_result[
                "context_recall"
            ],
            "faithfulness": evaluation["faithfulness"],
            "answer_relevancy": evaluation["relevancy"]
        })

        print("Completed:", question)

    return report

In [148]:
full_evaluation = run_light_evaluation()

print("\nEvaluation completed successfully! ✅")

Completed: What is Lean Construction?
Completed: What are the main principles of Lean Construction?
Completed: What is the purpose of Lean Construction?
Completed: What is the Last Planner System?
Completed: What is Value Stream Mapping?

Evaluation completed successfully! ✅


In [149]:
!pip install -q streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 36.6 MB/s eta 0:00:00


In [150]:
import streamlit as st

print("Streamlit installed successfully! ✅")

Streamlit installed successfully! ✅


In [151]:
test_answer, test_result = chat_with_agent(
    "What is Lean Construction?",
    []
)

print("Selected Tool:")
print(test_result["tool"])

print("\nFinal Answer:")
print(test_answer)

Selected Tool:
document_search

Final Answer:
Based on the provided documents, **Lean Construction** is an approach to construction management that applies manufacturing methods and lean principles (such as those from the Toyota Production System) to improve project performance, productivity, and delivery. 

Key aspects of Lean Construction covered in the text include:
* **Core Principles:** Value, Value Stream, Flow, Pull, and Perfection (Source: Page 3).
* **Foundations and Theory:** It contrasts with traditional construction methods by addressing their deficiencies, applying manufacturing methods, and utilizing a systems perspective to reduce or eliminate waste, move time, wait time, setup time, and process time (Source: Pages 2–3).
* **Implementation Systems:** It incorporates the Lean Project Delivery System, which covers phases like project definition, lean design, lean supply, lean assembly, production control, and work structuring—utilizing tools such as the Last Planner® Syste

In [153]:
%%writefile app.py

import streamlit as st
import json
import re

# ============================================================
# PAGE CONFIG
# ============================================================

st.set_page_config(
    page_title="AI Research & Document Assistant",
    page_icon="🤖",
    layout="wide"
)

# ============================================================
# TITLE
# ============================================================

st.title("🤖 AI Research & Document Assistant")

st.caption(
    "Agentic RAG system with document search, summarization, "
    "calculator, conversation memory, and source citations."
)

# ============================================================
# SESSION MEMORY
# ============================================================

if "conversation_history" not in st.session_state:
    st.session_state.conversation_history = []

# ============================================================
# SIDEBAR
# ============================================================

with st.sidebar:

    st.header("🛠️ Available Tools")

    st.write("📚 Document Search")
    st.write("📝 Summarizer")
    st.write("🧮 Calculator")

    st.divider()

    st.subheader("🧠 Agent Features")

    st.write("✅ LLM-based tool selection")
    st.write("✅ Semantic retrieval")
    st.write("✅ Conversation memory")
    st.write("✅ Source/page tracking")
    st.write("✅ LangGraph workflow")

    st.divider()

    if st.button("🗑️ Clear Conversation"):

        st.session_state.conversation_history = []

        st.rerun()

# ============================================================
# DISPLAY PREVIOUS CHAT
# ============================================================

for message in st.session_state.conversation_history:

    with st.chat_message("user"):
        st.write(message["user"])

    with st.chat_message("assistant"):
        st.write(message["assistant"])

# ============================================================
# USER INPUT
# ============================================================

query = st.chat_input(
    "Ask something about your research document..."
)

# ============================================================
# PROCESS QUERY
# ============================================================

if query:

    # Show user message

    with st.chat_message("user"):
        st.write(query)

    # Run agent

    with st.spinner("🤖 Agent is thinking..."):

        answer, result = chat_with_agent(
            query,
            st.session_state.conversation_history
        )

    # Show assistant answer

    with st.chat_message("assistant"):

        st.write(answer)

        # --------------------------------------------
        # Selected Tool
        # --------------------------------------------

        selected_tool = result.get("tool", "")

        if selected_tool:

            st.caption(
                f"🔧 Tool used: `{selected_tool}`"
            )

        # --------------------------------------------
        # Sources
        # --------------------------------------------

        tool_result = result.get(
            "tool_result",
            ""
        )

        if selected_tool == "document_search":

            try:

                sources = json.loads(tool_result)

                if sources:

                    st.markdown("### 📚 Sources")

                    for source in sources:

                        st.write(
                            f"📄 **{source['source']}** "
                            f"— Pages "
                            f"{source['start_page']}-"
                            f"{source['end_page']}"
                        )

            except Exception:

                pass

        elif selected_tool == "summarizer":

            try:

                summary_data = json.loads(
                    tool_result
                )

                sources = summary_data.get(
                    "sources",
                    []
                )

                if sources:

                    st.markdown("### 📚 Sources")

                    for source in sources:

                        st.write(
                            f"📄 **{source['source']}** "
                            f"— Pages "
                            f"{source['start_page']}-"
                            f"{source['end_page']}"
                        )

            except Exception:

                pass

    # ========================================================
    # SAVE CONVERSATION
    # ========================================================

    st.session_state.conversation_history.append(
        {
            "user": query,
            "assistant": answer
        }
    )

Writing app.py


In [154]:
import os

print("Current files:")

for file in os.listdir("/content"):
    print(file)

Current files:
.config
data
app.py
sample_data


In [155]:
import os
from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

print("Gemini API key loaded from Colab Secret ✅")

Gemini API key loaded from Colab Secret ✅


In [156]:
print("embedding_model:", "embedding_model" in globals())
print("collection:", "collection" in globals())
print("client:", "client" in globals())
print("agent:", "agent" in globals())
print("chat_with_agent:", "chat_with_agent" in globals())

embedding_model: True
collection: True
client: True
agent: True
chat_with_agent: True


In [157]:
%%writefile backend.py

import os
import re
import json
from typing import TypedDict

import chromadb
from sentence_transformers import SentenceTransformer
from google import genai
from langgraph.graph import StateGraph, START, END


# ============================================================
# CONFIGURATION
# ============================================================

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY not found.")

client = genai.Client(
    api_key=GEMINI_API_KEY
)

MODEL_NAME = "gemini-3.5-flash-lite"


# ============================================================
# EMBEDDING MODEL
# ============================================================

embedding_model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)


# ============================================================
# CHROMADB
# ============================================================

chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(
    name="research_documents"
)


# ============================================================
# DOCUMENT SEARCH
# ============================================================

def document_search(query, top_k=3):

    query_embedding = embedding_model.encode(
        query
    ).tolist()

    search_results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    retrieved_documents = search_results["documents"][0]
    retrieved_metadatas = search_results["metadatas"][0]

    results = []

    for doc, metadata in zip(
        retrieved_documents,
        retrieved_metadatas
    ):

        results.append({
            "text": doc,
            "source": metadata["source"],
            "start_page": metadata["start_page"],
            "end_page": metadata["end_page"]
        })

    return results


# ============================================================
# SUMMARIZER
# ============================================================

def summarize_text(text):

    prompt = f"""
You are a research document assistant.

Summarize the following document content clearly
and concisely.

Use ONLY the provided text.
Do not add outside information.

Document Content:

{text}

Provide:

1. Main idea
2. Important points
3. Key concepts
"""

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )

    return response.text


# ============================================================
# CALCULATOR
# ============================================================

def calculator(expression):

    allowed_chars = "0123456789+-*/().% "

    if not all(
        char in allowed_chars
        for char in expression
    ):
        return "Invalid mathematical expression."

    try:

        result = eval(
            expression,
            {"__builtins__": {}},
            {}
        )

        return result

    except Exception:

        return "Could not calculate the expression."


# ============================================================
# LLM TOOL SELECTION
# ============================================================

def llm_select_tool(query):

    prompt = f"""
You are a tool-selection system for an
AI Research Document Assistant.

Available tools:

1. document_search
Use for questions about the research document.

2. summarizer
Use when the user explicitly asks for a summary.

3. calculator
Use for mathematical calculations.

User Query:

{query}

Return ONLY one exact tool name:

document_search
summarizer
calculator
"""

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )

    selected_tool = response.text.strip().lower()

    valid_tools = [
        "document_search",
        "summarizer",
        "calculator"
    ]

    if selected_tool not in valid_tools:

        selected_tool = "document_search"

    return selected_tool


# ============================================================
# AGENT STATE
# ============================================================

class AgentState(TypedDict):

    query: str
    tool: str
    tool_result: str
    final_answer: str
    conversation_history: list


# ============================================================
# TOOL NODES
# ============================================================

def document_search_node(state: AgentState):

    results = document_search(
        state["query"]
    )

    formatted_results = []

    for result in results:

        formatted_results.append({

            "source": result["source"],

            "start_page":
                result["start_page"],

            "end_page":
                result["end_page"],

            "text":
                result["text"]

        })

    return {

        "tool_result":
            json.dumps(
                formatted_results,
                indent=2
            )

    }


def summarizer_node(state: AgentState):

    results = document_search(
        state["query"]
    )

    combined_text = "\n\n".join(

        result["text"]

        for result in results

    )

    summary = summarize_text(
        combined_text
    )

    sources = []

    for result in results:

        sources.append({

            "source":
                result["source"],

            "start_page":
                result["start_page"],

            "end_page":
                result["end_page"]

        })

    tool_output = {

        "summary":
            summary,

        "sources":
            sources

    }

    return {

        "tool_result":
            json.dumps(
                tool_output,
                indent=2
            )

    }


def calculator_node(state: AgentState):

    query = state["query"]

    match = re.search(

        r"(\d+(?:\.\d+)?)\s*%\s*of\s*(\d+(?:\.\d+)?)",

        query

    )

    if match:

        percentage = float(
            match.group(1)
        )

        number = float(
            match.group(2)
        )

        result = (
            percentage / 100
        ) * number

    else:

        expression = re.sub(

            r"[^0-9+\-*/().% ]",

            "",

            query

        )

        result = calculator(
            expression
        )

    tool_output = {

        "expression":
            query,

        "result":
            result

    }

    return {

        "tool_result":
            json.dumps(
                tool_output,
                indent=2
            )

    }


# ============================================================
# TOOL SELECTION NODE
# ============================================================

def select_tool(state: AgentState):

    tool = llm_select_tool(
        state["query"]
    )

    return {
        "tool": tool
    }


# ============================================================
# RUN SELECTED TOOL
# ============================================================

def run_selected_tool(state: AgentState):

    tool = state["tool"]

    if tool == "document_search":

        return document_search_node(
            state
        )

    elif tool == "summarizer":

        return summarizer_node(
            state
        )

    elif tool == "calculator":

        return calculator_node(
            state
        )

    return {

        "tool_result":
            json.dumps({

                "error":
                    "Unknown tool selected."

            })

    }


# ============================================================
# FINAL ANSWER
# ============================================================

def final_answer_node(state: AgentState):

    query = state["query"]

    tool = state["tool"]

    tool_result = state["tool_result"]

    history = state.get(
        "conversation_history",
        []
    )

    history_text = ""

    for message in history:

        history_text += (

            f"User: "
            f"{message['user']}\n"

            f"Assistant: "
            f"{message['assistant']}\n\n"

        )

    prompt = f"""
You are an AI Research Document Assistant.

Previous Conversation:

{history_text}

Current User Question:

{query}

Tool Used:

{tool}

Tool Result:

{tool_result}

Instructions:

1. Use previous conversation for follow-up questions.
2. Use the tool result to answer.
3. For document questions, use only document information.
4. Do not invent information.
5. Give a clear and concise answer.
6. Mention source pages when available.
"""

    response = client.models.generate_content(

        model=MODEL_NAME,

        contents=prompt

    )

    return {

        "final_answer":
            response.text

    }


# ============================================================
# BUILD LANGGRAPH
# ============================================================

graph = StateGraph(
    AgentState
)

graph.add_node(
    "select_tool",
    select_tool
)

graph.add_node(
    "run_tool",
    run_selected_tool
)

graph.add_node(
    "final_answer",
    final_answer_node
)

graph.add_edge(
    START,
    "select_tool"
)

graph.add_edge(
    "select_tool",
    "run_tool"
)

graph.add_edge(
    "run_tool",
    "final_answer"
)

graph.add_edge(
    "final_answer",
    END
)

agent = graph.compile()


# ============================================================
# CHAT FUNCTION
# ============================================================

def chat_with_agent(
    query,
    conversation_history
):

    initial_state = {

        "query":
            query,

        "tool":
            "",

        "tool_result":
            "",

        "final_answer":
            "",

        "conversation_history":
            conversation_history

    }

    result = agent.invoke(
        initial_state
    )

    conversation_history.append({

        "user":
            query,

        "assistant":
            result["final_answer"]

    })

    return (

        result["final_answer"],

        result

    )

Writing backend.py


In [158]:
import os

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

print("API key ready ✅")
print("PDF exists:", os.path.exists("/content/data/1.pdf"))

API key ready ✅
PDF exists: True


In [159]:
%%writefile backend.py

import os
import re
import json
from pathlib import Path
from typing import TypedDict

import chromadb
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from google import genai
from langgraph.graph import StateGraph, START, END


# ============================================================
# CONFIG
# ============================================================

PDF_PATH = Path("/content/data/1.pdf")

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY not found.")

client = genai.Client(
    api_key=GEMINI_API_KEY
)

MODEL_NAME = "gemini-3.5-flash-lite"


# ============================================================
# EMBEDDING MODEL
# ============================================================

embedding_model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)


# ============================================================
# PDF EXTRACTION
# ============================================================

def clean_text(text):

    text = re.sub(r"\s+", " ", text)

    return text.strip()


def load_pdf():

    reader = PdfReader(
        str(PDF_PATH)
    )

    documents = []

    for page_number, page in enumerate(
        reader.pages,
        start=1
    ):

        text = page.extract_text()

        if text:

            documents.append({

                "text":
                    clean_text(text),

                "page":
                    page_number,

                "source":
                    PDF_PATH.name

            })

    return documents


# ============================================================
# CHUNKING
# ============================================================

def create_chunks(
    documents,
    chunk_size=500,
    overlap=50
):

    chunks = []

    all_words = []

    for doc in documents:

        words = doc["text"].split()

        for word in words:

            all_words.append({

                "word":
                    word,

                "page":
                    doc["page"],

                "source":
                    doc["source"]

            })

    start = 0

    chunk_id = 0

    while start < len(all_words):

        end = min(
            start + chunk_size,
            len(all_words)
        )

        chunk_items = all_words[
            start:end
        ]

        chunk_text = " ".join(

            item["word"]

            for item in chunk_items

        )

        chunks.append({

            "id":
                f"chunk_{chunk_id}",

            "text":
                chunk_text,

            "source":
                chunk_items[0]["source"],

            "start_page":
                chunk_items[0]["page"],

            "end_page":
                chunk_items[-1]["page"]

        })

        chunk_id += 1

        start += (
            chunk_size - overlap
        )

    return chunks


# ============================================================
# CHROMADB
# ============================================================

chroma_client = chromadb.PersistentClient(
    path="/content/chroma_db"
)

collection = chroma_client.get_or_create_collection(
    name="research_documents"
)


# ============================================================
# INDEX DOCUMENT
# ============================================================

def build_index():

    documents = load_pdf()

    chunks = create_chunks(
        documents
    )

    existing = collection.count()

    if existing == 0:

        texts = [
            chunk["text"]
            for chunk in chunks
        ]

        embeddings = embedding_model.encode(
            texts,
            show_progress_bar=False
        )

        ids = [
            chunk["id"]
            for chunk in chunks
        ]

        metadatas = [

            {
                "source":
                    chunk["source"],

                "start_page":
                    chunk["start_page"],

                "end_page":
                    chunk["end_page"]

            }

            for chunk in chunks

        ]

        collection.add(

            ids=ids,

            documents=texts,

            embeddings=embeddings.tolist(),

            metadatas=metadatas

        )

    return collection.count()


chunk_count = build_index()


# ============================================================
# DOCUMENT SEARCH
# ============================================================

def document_search(
    query,
    top_k=3
):

    query_embedding = embedding_model.encode(
        query
    ).tolist()

    results = collection.query(

        query_embeddings=[
            query_embedding
        ],

        n_results=top_k

    )

    documents = results["documents"][0]

    metadatas = results["metadatas"][0]

    output = []

    for doc, metadata in zip(
        documents,
        metadatas
    ):

        output.append({

            "text":
                doc,

            "source":
                metadata["source"],

            "start_page":
                metadata["start_page"],

            "end_page":
                metadata["end_page"]

        })

    return output


# ============================================================
# SUMMARIZER
# ============================================================

def summarize_text(text):

    prompt = f"""
You are a research document assistant.

Summarize the following document content.

Use ONLY the provided content.

Document Content:

{text}

Provide:

1. Main idea
2. Important points
3. Key concepts
"""

    response = client.models.generate_content(

        model=MODEL_NAME,

        contents=prompt

    )

    return response.text


# ============================================================
# CALCULATOR
# ============================================================

def calculator(expression):

    allowed_chars = (
        "0123456789+-*/().% "
    )

    if not all(
        char in allowed_chars
        for char in expression
    ):

        return "Invalid mathematical expression."

    try:

        return eval(
            expression,
            {"__builtins__": {}},
            {}
        )

    except Exception:

        return "Could not calculate the expression."


# ============================================================
# TOOL SELECTION
# ============================================================

def llm_select_tool(query):

    prompt = f"""
You are a tool-selection system.

Available tools:

document_search
- Questions about the research document.

summarizer
- User explicitly asks for a summary.

calculator
- Mathematical calculations.

User Query:

{query}

Return ONLY one:

document_search
summarizer
calculator
"""

    response = client.models.generate_content(

        model=MODEL_NAME,

        contents=prompt

    )

    tool = response.text.strip().lower()

    valid_tools = [
        "document_search",
        "summarizer",
        "calculator"
    ]

    if tool not in valid_tools:

        tool = "document_search"

    return tool


# ============================================================
# AGENT STATE
# ============================================================

class AgentState(TypedDict):

    query: str
    tool: str
    tool_result: str
    final_answer: str
    conversation_history: list


# ============================================================
# TOOL NODES
# ============================================================

def document_search_node(
    state
):

    results = document_search(
        state["query"]
    )

    output = []

    for result in results:

        output.append({

            "source":
                result["source"],

            "start_page":
                result["start_page"],

            "end_page":
                result["end_page"],

            "text":
                result["text"]

        })

    return {

        "tool_result":
            json.dumps(
                output,
                indent=2
            )

    }


def summarizer_node(
    state
):

    results = document_search(
        state["query"]
    )

    combined_text = "\n\n".join(

        result["text"]

        for result in results

    )

    summary = summarize_text(
        combined_text
    )

    sources = []

    for result in results:

        sources.append({

            "source":
                result["source"],

            "start_page":
                result["start_page"],

            "end_page":
                result["end_page"]

        })

    return {

        "tool_result":
            json.dumps({

                "summary":
                    summary,

                "sources":
                    sources

            })

    }


def calculator_node(
    state
):

    query = state["query"]

    match = re.search(

        r"(\d+(?:\.\d+)?)\s*%\s*of\s*(\d+(?:\.\d+)?)",

        query

    )

    if match:

        percentage = float(
            match.group(1)
        )

        number = float(
            match.group(2)
        )

        result = (
            percentage / 100
        ) * number

    else:

        expression = re.sub(
            r"[^0-9+\-*/().% ]",
            "",
            query
        )

        result = calculator(
            expression
        )

    return {

        "tool_result":
            json.dumps({

                "expression":
                    query,

                "result":
                    result

            })

    }


# ============================================================
# SELECT TOOL
# ============================================================

def select_tool(state):

    return {

        "tool":
            llm_select_tool(
                state["query"]
            )

    }


# ============================================================
# RUN TOOL
# ============================================================

def run_selected_tool(
    state
):

    tool = state["tool"]

    if tool == "document_search":

        return document_search_node(
            state
        )

    if tool == "summarizer":

        return summarizer_node(
            state
        )

    if tool == "calculator":

        return calculator_node(
            state
        )

    return {

        "tool_result":
            json.dumps({

                "error":
                    "Unknown tool."

            })

    }


# ============================================================
# FINAL ANSWER
# ============================================================

def final_answer_node(
    state
):

    history = state.get(
        "conversation_history",
        []
    )

    history_text = ""

    for message in history:

        history_text += (
            f"User: {message['user']}\n"
            f"Assistant: {message['assistant']}\n\n"
        )

    prompt = f"""
You are an AI Research Document Assistant.

Previous Conversation:

{history_text}

Current Question:

{state["query"]}

Selected Tool:

{state["tool"]}

Tool Result:

{state["tool_result"]}

Instructions:

1. Use previous conversation for context.
2. Use the tool result to answer.
3. For document questions, use ONLY retrieved document information.
4. Do not invent facts.
5. Answer clearly and concisely.
6. Mention source pages when available.
"""

    response = client.models.generate_content(

        model=MODEL_NAME,

        contents=prompt

    )

    return {

        "final_answer":
            response.text

    }


# ============================================================
# LANGGRAPH
# ============================================================

graph = StateGraph(
    AgentState
)

graph.add_node(
    "select_tool",
    select_tool
)

graph.add_node(
    "run_tool",
    run_selected_tool
)

graph.add_node(
    "final_answer",
    final_answer_node
)

graph.add_edge(
    START,
    "select_tool"
)

graph.add_edge(
    "select_tool",
    "run_tool"
)

graph.add_edge(
    "run_tool",
    "final_answer"
)

graph.add_edge(
    "final_answer",
    END
)

agent = graph.compile()


# ============================================================
# CHAT FUNCTION
# ============================================================

def chat_with_agent(
    query,
    conversation_history
):

    initial_state = {

        "query":
            query,

        "tool":
            "",

        "tool_result":
            "",

        "final_answer":
            "",

        "conversation_history":
            conversation_history.copy()

    }

    result = agent.invoke(
        initial_state
    )

    return (
        result["final_answer"],
        result
    )


print(
    f"Backend ready ✅ | "
    f"Indexed chunks: {chunk_count}"
)

Overwriting backend.py


In [160]:
import sys
import importlib

sys.path.append("/content")

import backend

print("\nIndexed chunks:", backend.chunk_count)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Backend ready ✅ | Indexed chunks: 8

Indexed chunks: 8


In [161]:
answer, result = backend.chat_with_agent(
    "What is Lean Construction?",
    []
)

print("Selected Tool:")
print(result["tool"])

print("\nAnswer:")
print(answer)

Selected Tool:
document_search

Answer:
Based on the retrieved documents, **Lean Construction** is an approach that applies manufacturing methods (originating from production systems like Toyota's) to the construction industry to improve productivity and performance compared to traditional construction methods (source `1.pdf`, pages 1, 3, and 45). 

Key aspects of Lean Construction include:
* **Core Principles:** Focusing on value, the value stream, flow, pull, and perfection (source `1.pdf`, pages 3 and 4).
* **Philosophical Difference:** It differs fundamentally from traditional construction by addressing deficiencies such as low productivity, crisis orientation, and ineffective management practices, shifting the focus toward continuous learning, waste reduction, and systems perspectives (source `1.pdf`, pages 3 and 4).
* **Implementation:** It encompasses specific frameworks and tools like the Lean Project Delivery System, Target-Value Design, Set-Based Design, the Last Planner® Sys

In [162]:
results = backend.document_search(
    "What is Lean Construction?",
    top_k=3
)

for r in results:
    print(
        f"{r['source']} | "
        f"Pages {r['start_page']}-{r['end_page']}"
    )

1.pdf | Pages 4-6
1.pdf | Pages 1-4
1.pdf | Pages 16-17


In [163]:
from pathlib import Path

backend_path = Path("/content/backend.py")

code = backend_path.read_text()

start = code.index(
    "# ============================================================\n"
    "# FINAL ANSWER\n"
)

end = code.index(
    "# ============================================================\n"
    "# LANGGRAPH\n"
)

new_final_answer = r'''# ============================================================
# FINAL ANSWER
# ============================================================

def final_answer_node(state):

    history = state.get(
        "conversation_history",
        []
    )

    history_text = ""

    for message in history:

        history_text += (
            f"User: {message['user']}\n"
            f"Assistant: {message['assistant']}\n\n"
        )

    # --------------------------------------------------------
    # Extract exact source information from tool result
    # --------------------------------------------------------

    source_information = []

    try:

        parsed_result = json.loads(
            state["tool_result"]
        )

        if isinstance(parsed_result, list):

            for item in parsed_result:

                source_information.append(
                    f"{item['source']} | "
                    f"Pages {item['start_page']}-"
                    f"{item['end_page']}"
                )

        elif isinstance(parsed_result, dict):

            for item in parsed_result.get(
                "sources",
                []
            ):

                source_information.append(
                    f"{item['source']} | "
                    f"Pages {item['start_page']}-"
                    f"{item['end_page']}"
                )

    except Exception:

        pass

    verified_sources = "\n".join(
        source_information
    )

    prompt = f"""
You are an AI Research Document Assistant.

Previous Conversation:

{history_text}

Current Question:

{state["query"]}

Selected Tool:

{state["tool"]}

Tool Result:

{state["tool_result"]}

Verified Sources:

{verified_sources}

Instructions:

1. Answer the user's question using the tool result.
2. For document questions, use ONLY the retrieved document content.
3. Do NOT invent facts.
4. Do NOT invent page numbers.
5. Do NOT create or modify source citations.
6. The "Verified Sources" section contains the ONLY valid source/page information.
7. Give a clear and concise answer.
8. Do not mention sources inside the answer unless necessary.
9. Source information will be displayed separately by the application.
"""

    response = client.models.generate_content(

        model=MODEL_NAME,

        contents=prompt

    )

    return {

        "final_answer":
            response.text

    }


'''

backend_path.write_text(
    code[:start] +
    new_final_answer +
    code[end:]
)

print("final_answer_node updated successfully ✅")

final_answer_node updated successfully ✅


In [164]:
import importlib

import backend

importlib.reload(backend)

print(
    "Backend reloaded successfully ✅"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Backend ready ✅ | Indexed chunks: 8
Backend reloaded successfully ✅


In [165]:
answer, result = backend.chat_with_agent(
    "What is Lean Construction?",
    []
)

print("Selected Tool:")
print(result["tool"])

print("\nAnswer:")
print(answer)

print("\nVerified Sources:")

for r in backend.document_search(
    "What is Lean Construction?",
    top_k=3
):

    print(
        f"{r['source']} | "
        f"Pages {r['start_page']}-"
        f"{r['end_page']}"
    )

Selected Tool:
document_search

Answer:
Based on the provided document, **Lean Construction** is an approach modeled after manufacturing methods (such as the Toyota Production System) adapted for the construction industry. It contrasts with traditional construction methods and focuses on improving productivity, performance, and project delivery through specific principles, systems, and tools.

Key characteristics and components of Lean Construction include:

* **Core Lean Principles:** 
  * **Value:** Defining value from the customer's perspective.
  * **Value Stream & Value Stream Mapping:** Identifying and mapping all the steps in a process to eliminate waste.
  * **Flow:** Ensuring that work flows smoothly and uninterrupted through the production process.
  * **Pull:** Producing work only when it is needed by the downstream process (using pull systems instead of push systems).
  * **Perfection:** Continuously striving for continuous improvement.

* **System Components & Focus:**
  *

In [166]:
from backend import chat_with_agent

In [167]:
%%writefile app.py

import streamlit as st
import json

from backend import chat_with_agent


# ============================================================
# PAGE CONFIG
# ============================================================

st.set_page_config(
    page_title="AI Research & Document Assistant",
    page_icon="🤖",
    layout="wide"
)


# ============================================================
# TITLE
# ============================================================

st.title("🤖 AI Research & Document Assistant")

st.caption(
    "Agentic RAG system with document search, summarization, "
    "calculator, conversation memory, and source citations."
)


# ============================================================
# SESSION MEMORY
# ============================================================

if "conversation_history" not in st.session_state:

    st.session_state.conversation_history = []


# ============================================================
# SIDEBAR
# ============================================================

with st.sidebar:

    st.header("🛠️ Available Tools")

    st.write("📚 Document Search")
    st.write("📝 Summarizer")
    st.write("🧮 Calculator")

    st.divider()

    st.subheader("🧠 Agent Features")

    st.write("✅ LLM-based tool selection")
    st.write("✅ Semantic retrieval")
    st.write("✅ Conversation memory")
    st.write("✅ Source/page tracking")
    st.write("✅ LangGraph workflow")

    st.divider()

    if st.button("🗑️ Clear Conversation"):

        st.session_state.conversation_history = []

        st.rerun()


# ============================================================
# DISPLAY CHAT HISTORY
# ============================================================

for message in st.session_state.conversation_history:

    with st.chat_message("user"):

        st.write(message["user"])

    with st.chat_message("assistant"):

        st.write(message["assistant"])


# ============================================================
# CHAT INPUT
# ============================================================

query = st.chat_input(
    "Ask something about your research document..."
)


# ============================================================
# PROCESS QUERY
# ============================================================

if query:

    # --------------------------------------------------------
    # Display user message
    # --------------------------------------------------------

    with st.chat_message("user"):

        st.write(query)


    # --------------------------------------------------------
    # Run Agent
    # --------------------------------------------------------

    with st.spinner("🤖 Agent is thinking..."):

        answer, result = chat_with_agent(

            query,

            st.session_state.conversation_history

        )


    # --------------------------------------------------------
    # Display Assistant Answer
    # --------------------------------------------------------

    with st.chat_message("assistant"):

        st.write(answer)


        # ----------------------------------------------------
        # Tool Used
        # ----------------------------------------------------

        selected_tool = result.get(
            "tool",
            ""
        )

        if selected_tool:

            st.caption(
                f"🔧 Tool used: `{selected_tool}`"
            )


        # ----------------------------------------------------
        # Sources
        # ----------------------------------------------------

        tool_result = result.get(
            "tool_result",
            ""
        )


        if selected_tool == "document_search":

            try:

                sources = json.loads(
                    tool_result
                )

                if sources:

                    st.markdown(
                        "### 📚 Sources"
                    )

                    for source in sources:

                        st.write(
                            f"📄 **{source['source']}** "
                            f"— Pages "
                            f"{source['start_page']}-"
                            f"{source['end_page']}"
                        )

            except Exception:

                pass


        elif selected_tool == "summarizer":

            try:

                summary_data = json.loads(
                    tool_result
                )

                sources = summary_data.get(
                    "sources",
                    []
                )

                if sources:

                    st.markdown(
                        "### 📚 Sources"
                    )

                    for source in sources:

                        st.write(
                            f"📄 **{source['source']}** "
                            f"— Pages "
                            f"{source['start_page']}-"
                            f"{source['end_page']}"
                        )

            except Exception:

                pass


    # ========================================================
    # SAVE CONVERSATION
    # ========================================================

    st.session_state.conversation_history.append(

        {
            "user":
                query,

            "assistant":
                answer
        }

    )

Overwriting app.py


In [168]:
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇
added 22 packages in 2s
⠇
⠇3 packages are looking for funding
⠇  run `npm fund` for details
⠇npm notice
npm notice New major version of npm available! 10.8.2 -> 12.0.2
npm notice Changelog: https://github.com/npm/cli/releases/tag/v12.0.2
npm notice To update run: npm install -g npm@12.0.2
npm notice
⠇

In [ ]:
# ============================================
# Streamlit + Cloudflare Tunnel
# ============================================

import os
import time
import subprocess

print("1️⃣ Stopping old Streamlit and LocalTunnel...")

os.system("pkill -f streamlit || true")
os.system("pkill -f 'lt --port 8501' || true")
os.system("pkill -f cloudflared || true")

time.sleep(2)

print("2️⃣ Starting Streamlit...")

os.system(
    "streamlit run app.py "
    "--server.port 8501 "
    "--server.address 0.0.0.0 "
    "> /content/streamlit.log 2>&1 &"
)

time.sleep(5)

print("3️⃣ Checking Streamlit...")

os.system("cat /content/streamlit.log")

print("\n4️⃣ Installing Cloudflare Tunnel...")

if not os.path.exists("/content/cloudflared"):
    os.system(
        "wget -q "
        "https://github.com/cloudflare/cloudflared/releases/latest/download/"
        "cloudflared-linux-amd64 "
        "-O /content/cloudflared"
    )
    os.system("chmod +x /content/cloudflared")

print("5️⃣ Starting Cloudflare Tunnel...")

os.system(
    "nohup /content/cloudflared tunnel "
    "--url http://localhost:8501 "
    "--no-autoupdate "
    "> /content/cloudflared.log 2>&1 &"
)

time.sleep(8)

print("\n============================================")
print("🚀 YOUR STREAMLIT PUBLIC URL")
print("============================================\n")

os.system("grep -o 'https://[-a-zA-Z0-9]*\\.trycloudflare\\.com' /content/cloudflared.log | head -1")

print("\n============================================")
print("Open the https://....trycloudflare.com URL above")
print("============================================")

your url is: https://nine-pears-strive.loca.lt
